# Supporting context vs equal-size random additions, plan4_dev, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

`plan4_dev` only. Do **not** load `plan4_hold` or `plan4_fresh`. Do not resume the interface factorial.

A is not regenerated. Caches are rebuilt from the recorded token IDs in `results_stage_plan4_iface/a_artifacts.jsonl`. The one capacity miss stays as recorded.

Baseline: `cons_instr_sink` (result lines, first four positions, consecutive RoPE, framing sentence).

**Run cell 4 first** (`--phase construct`). That freezes the instruction paragraph, calculation block, and three random selections. Read the printed inspection. If `all_ok` is false, stop.

If construction passes, run cell 5 (`--phase compare`): eight conditions, 720 receiver evaluations.

Download `/kaggle/working/results_stage_plan4_support`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_support.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_support.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_support.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_support.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("plan4_dev", sum(1 for _ in (REPO / "data" / "plan4_dev.jsonl").open(encoding="utf-8")))
assert not (REPO / "data" / "plan4_hold.jsonl").samefile(REPO / "data" / "plan4_dev.jsonl")
art = REPO / "results_stage_plan4_iface" / "a_artifacts.jsonl"
print("frozen A artifacts", art.is_file(), art)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_support")
print("Run construct first. Only then compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_support", "--split", "plan4_dev", "--phase", "construct"])
print("plan4 support construct exit", rc)
path = WORK / "results_stage_plan4_support" / "construction.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_support" / "run" / "construction.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    slim = {k: data.get(k) for k in ("n", "n_ok", "all_ok", "mean_n_extra", "mean_overlap_frac", "n_a_errors_preserved", "sampling_modes", "inspect", "failures", "frozen_rule")}
    print(json.dumps(slim, indent=2)[:16000])

In [ ]:
src = WORK / "results_stage_plan4_support" / "construction.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_support" / "run" / "construction.json"
construction = json.loads(src.read_text(encoding="utf-8"))
if not construction.get("all_ok"):
    raise RuntimeError("Construction failed. Do not run the eight-condition comparison yet.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_support", "--split", "plan4_dev", "--phase", "compare"])
print("plan4 support compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_support"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_support" / "run"
print("reading", src)
path = src / "summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    keep = {k: data.get(k) for k in ("n_rows", "n_examples", "receiver", "family_clustered_eligibility", "decision", "reconstruction_check", "lead")}
    print(json.dumps(keep, indent=2)[:24000])